# Stage 4.5-4.6 (rebuilt): Double ML elasticity + asymmetry + final table

Self-contained: reads `../data/panel.parquet`, rebuilds the features, and writes `../data/elasticity_final.csv`. Run top to bottom. Takes about 2-3 minutes.

## 1. Load the panel and build features

In [1]:
import pandas as pd, numpy as np
import xgboost as xgb
import statsmodels.api as sm

panel = (pd.read_parquet("../data/panel.parquet")
         .sort_values(["StockCode", "Week"]).reset_index(drop=True))
g = panel.groupby("StockCode")

# Same features as notebook 05 (all use past weeks only, except the treatment itself)
panel["lag1_log_units"]  = g["log_units"].shift(1)
panel["roll4_log_units"] = g["log_units"].transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())
panel["weeks_gap"]       = g["Week"].diff().dt.days / 7
panel["cat_code"]        = panel["category"].astype("category").cat.codes

# Treatment: log price relative to the product's own prior 8-week average
panel["ref_logp8"] = g["log_price"].transform(lambda s: s.shift(1).rolling(8, min_periods=2).mean())
panel["rel_price"] = panel["log_price"] - panel["ref_logp8"]

# Order-size signals (lagged only)
panel["log_upl"]   = np.log(panel["units"] / panel["n_lines"])
panel["log_lines"] = np.log(panel["n_lines"])
g = panel.groupby("StockCode")
panel["lag1_log_upl"]   = g["log_upl"].shift(1)
panel["roll4_log_upl"]  = g["log_upl"].transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())
panel["lag1_log_lines"] = g["log_lines"].shift(1)

# History and volatility
panel["roll8_log_units"] = g["log_units"].transform(lambda s: s.shift(1).rolling(8, min_periods=2).mean())
panel["std4_log_units"]  = g["log_units"].transform(lambda s: s.shift(1).rolling(4, min_periods=2).std())
panel["max8_log_units"]  = g["log_units"].transform(lambda s: s.shift(1).rolling(8, min_periods=1).max())
panel["obs_so_far"]      = g.cumcount()

# Calendar
xmas = pd.to_datetime(panel["Week"].dt.year.astype(str) + "-12-25")
panel["weeks_to_xmas"] = ((xmas - panel["Week"]).dt.days / 7).clip(-10, 52)

# Market and category momentum (previous-week demand)
panel["mkt_lag1"] = panel.groupby("Week")["lag1_log_units"].transform("mean")
panel["cat_lag1"] = panel.groupby(["category", "Week"])["lag1_log_units"].transform("mean")

print(panel.shape)

(66694, 37)


## 2. Cross-fitted residuals (nuisance models)

In [2]:
XD = ["ref_logp8", "lag1_log_units", "roll4_log_units", "roll8_log_units", "std4_log_units",
      "max8_log_units", "lag1_log_upl", "roll4_log_upl", "lag1_log_lines", "weeks_gap",
      "obs_so_far", "week_of_year", "month", "is_q4", "weeks_to_xmas", "cat_code",
      "mkt_lag1", "cat_lag1"]

missing = [c for c in XD + ["rel_price", "log_units"] if c not in panel.columns]
assert not missing, f"Missing columns: {missing}"

dml = panel.dropna(subset=["rel_price", "log_units"]).copy()
weeks = np.sort(dml["Week"].unique())
K = 5
block_of = {w: i * K // len(weeks) for i, w in enumerate(weeks)}
dml["blk"] = dml["Week"].map(block_of)
print(dml.shape)          # expect (63294, ...)

def cross_fit(target):
    pred = np.zeros(len(dml))
    for b in range(K):
        tr_i, te_i = (dml["blk"] != b).values, (dml["blk"] == b).values
        m = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=10, random_state=42)
        m.fit(dml.loc[tr_i, XD], dml.loc[tr_i, target])
        pred[te_i] = m.predict(dml.loc[te_i, XD])
    return pred

y_res = dml["log_units"] - cross_fit("log_units")
t_res = dml["rel_price"] - cross_fit("rel_price")
dml["y_res"], dml["t_res"] = y_res.values, t_res.values

print("Share of log_units explained by controls:", round(1 - y_res.var() / dml["log_units"].var(), 3))
print("Share of rel_price explained by controls:", round(1 - t_res.var() / dml["rel_price"].var(), 3))

(63294, 38)
Share of log_units explained by controls: 0.269
Share of rel_price explained by controls: 0.006


## 3. Overall and by-category elasticity

In [3]:
groups = dml["StockCode"].astype("category").cat.codes

fit = sm.OLS(y_res, t_res).fit(cov_type="cluster", cov_kwds={"groups": groups})
th, se = fit.params.iloc[0], fit.bse.iloc[0]
print(f"DML elasticity: {th:.3f}  (SE {se:.3f}, 95% CI {th - 1.96*se:.3f} to {th + 1.96*se:.3f})")
print("Expected from the earlier run: about -1.688 (SE 0.020)")

rows = []
for cat in dml["category"].unique():
    mk = (dml["category"] == cat).values
    if dml.loc[mk, "StockCode"].nunique() < 20:
        continue
    f = sm.OLS(y_res[mk], t_res[mk]).fit(cov_type="cluster", cov_kwds={"groups": groups[mk]})
    rows.append((cat, f.params.iloc[0], f.bse.iloc[0], dml.loc[mk, "StockCode"].nunique()))

bycat = pd.DataFrame(rows, columns=["category", "elasticity", "se", "products"]).sort_values("elasticity")
bycat["ci_low"]  = bycat["elasticity"] - 1.96 * bycat["se"]
bycat["ci_high"] = bycat["elasticity"] + 1.96 * bycat["se"]
print(bycat.round(3))

DML elasticity: -1.688  (SE 0.020, 95% CI -1.727 to -1.649)
Expected from the earlier run: about -1.688 (SE 0.020)
                   category  elasticity     se  products  ci_low  ci_high
1                Stationery      -2.166  0.122        67  -2.405   -1.926
2             Paper & Party      -1.965  0.094       138  -2.150   -1.780
9               Toys & Kids      -1.938  0.176        28  -2.283   -1.594
10     Christmas & Seasonal      -1.839  0.116        63  -2.066   -1.611
3          Garden & Outdoor      -1.788  0.076        99  -1.938   -1.638
7        Lighting & Candles      -1.719  0.052       168  -1.821   -1.617
0                     Other      -1.640  0.035       516  -1.709   -1.570
8                Home Decor      -1.619  0.050       216  -1.717   -1.520
5            Bags & Storage      -1.617  0.058       160  -1.731   -1.503
4          Kitchen & Dining      -1.548  0.053       178  -1.652   -1.444
6   Jewellery & Accessories      -1.307  0.121        67  -1.545   -1.0

## 4. Asymmetry multipliers (cuts vs. hikes)

In [4]:
# Asymmetry: separate slopes for price cuts and hikes inside the simulator's +/-20% range
lo_cut, hi_hike = np.log(0.80), np.log(1.20)

def slope(mask):
    m = mask.values
    X = sm.add_constant(dml.loc[m, "t_res"])
    f = sm.OLS(dml.loc[m, "y_res"], X).fit(cov_type="cluster", cov_kwds={"groups": groups[m]})
    return f.params["t_res"], f.bse["t_res"], int(m.sum())

th_cut,  se_cut,  n_cut  = slope((dml["rel_price"] >= lo_cut) & (dml["rel_price"] < 0))
th_hike, se_hike, n_hike = slope((dml["rel_price"] > 0) & (dml["rel_price"] <= hi_hike))

print(f"Cuts  (0% to -20%): slope {th_cut:.2f} (SE {se_cut:.2f}), n={n_cut}")
print(f"Hikes (0% to +20%): slope {th_hike:.2f} (SE {se_hike:.2f}), n={n_hike}")

k_cut, k_hike = th_cut / th, th_hike / th
print(f"k_cut = {k_cut:.2f}, k_hike = {k_hike:.2f}   (overall DML {th:.2f})")

Cuts  (0% to -20%): slope -1.85 (SE 0.09), n=18556
Hikes (0% to +20%): slope -1.62 (SE 0.12), n=6743
k_cut = 1.10, k_hike = 0.96   (overall DML -1.69)


## 5. Write the final table for Power BI

In [5]:
# Final table for Power BI: DML + fixed-effects elasticities side by side, plus asymmetry multipliers
try:
    fe = (pd.read_csv("../data/elasticity_by_category.csv")[["category", "elasticity"]]
            .rename(columns={"elasticity": "fe_elasticity"}))
except FileNotFoundError:
    fe = pd.DataFrame({"category": bycat["category"], "fe_elasticity": np.nan})
    print("elasticity_by_category.csv not found: fe_elasticity left empty")

final = (bycat.rename(columns={"elasticity": "dml_elasticity", "se": "dml_se",
                               "ci_low": "dml_ci_low", "ci_high": "dml_ci_high"})
              .merge(fe, on="category", how="left"))
final["k_cut"], final["k_hike"] = round(k_cut, 3), round(k_hike, 3)

final.to_csv("../data/elasticity_final.csv", index=False)
print(final.round(3))

                   category  dml_elasticity  dml_se  products  dml_ci_low  \
0                Stationery          -2.166   0.122        67      -2.405   
1             Paper & Party          -1.965   0.094       138      -2.150   
2               Toys & Kids          -1.938   0.176        28      -2.283   
3      Christmas & Seasonal          -1.839   0.116        63      -2.066   
4          Garden & Outdoor          -1.788   0.076        99      -1.938   
5        Lighting & Candles          -1.719   0.052       168      -1.821   
6                     Other          -1.640   0.035       516      -1.709   
7                Home Decor          -1.619   0.050       216      -1.717   
8            Bags & Storage          -1.617   0.058       160      -1.731   
9          Kitchen & Dining          -1.548   0.053       178      -1.652   
10  Jewellery & Accessories          -1.307   0.121        67      -1.545   

    dml_ci_high  fe_elasticity  k_cut  k_hike  
0        -1.926         -1.

In [6]:

panel["median_price"] = panel.groupby("StockCode")["price"].transform("median")
panel["rev_model"] = panel["units"] * panel["price"]

cols = ["StockCode", "desc", "category", "Week", "units", "rev_model",
        "n_lines", "price", "median_price"]
panel[cols].to_csv("../data/powerbi_panel.csv", index=False)
print(len(panel), panel["Week"].min(), panel["Week"].max(), panel["rev_model"].sum().round(0))

66694 2009-11-30 00:00:00 2011-12-05 00:00:00 1612652.0


In [7]:
import pandas as pd
p = pd.read_parquet("../data/panel.parquet")
p["median_price"] = p.groupby("StockCode")["price"].transform("median")
print(1 - 0.6 * (p["median_price"] * p["units"]).sum() / (p["price"] * p["units"]).sum())

0.3146687361149718
